# Variable de control
##### María Fernanda Gómez Flores
##### Miércoles 21 de octubre de 2026

In [1]:
import numpy as np
from scipy.stats import norm

### Parámetros

In [2]:
S0 = 100
r = 0.05
sigma = 0.20
K = 100
T = 1
M = 100   # rutas
n = 252   # pasos
dt = T/n

### Precio teórico de la call geométrica

Primero saco el valor esperado de la geométrica para usarlo como variable de control.

In [3]:
mu_G = np.log(S0) + (r-0.5*sigma**2)*T*(n+1)/(2*n)
var_G = sigma**2*T*(n+1)*(2*n+1)/(6*n**2)

d1 = (mu_G + var_G - np.log(K))/np.sqrt(var_G)
d2 = (mu_G - np.log(K))/np.sqrt(var_G)

EX = np.exp(-r*T)*(np.exp(mu_G+var_G/2)*norm.cdf(d1) - K*norm.cdf(d2))
EX

np.float64(5.565508831349208)

### Simulación de 100 rutas

In [4]:
np.random.seed(32)

trayectorias = []

for i in range(M):
    precios = []
    S = S0

    for j in range(n):
        Z = np.random.normal(0, 1)
        S = S*np.exp((r-0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z)
        precios.append(S)

    trayectorias.append(precios)

len(trayectorias)

100

### Call aritmética y geométrica

Saco el promedio de cada ruta y calculo los payoffs de las dos calls.

In [5]:
Y = []   # call aritmética
X = []   # call geométrica

for precios in trayectorias:
    media_A = np.mean(precios)
    media_G = np.exp(np.mean(np.log(precios)))

    Y.append(np.exp(-r*T)*max(media_A-K, 0))
    X.append(np.exp(-r*T)*max(media_G-K, 0))

Y = np.array(Y)
X = np.array(X)

print("Call aritmética:", round(np.mean(Y), 4))
print("Call geométrica simulada:", round(np.mean(X), 4))

Call aritmética: 5.9677
Call geométrica simulada: 5.6981


### Variable de control

$$\hat b=\frac{\sum(X_i-\bar X)(Y_i-\bar Y)}{\sum(X_i-\bar X)^2}$$

In [6]:
x_media = np.mean(X)
y_media = np.mean(Y)

b = np.sum((X-x_media)*(Y-y_media))/np.sum((X-x_media)**2)
b

np.float64(1.043987368708223)

Aplico la misma fórmula que hicimos a mano:

$$Y_i(\hat b)=Y_i-\hat b(X_i-E[X])$$

In [7]:
Y_control = Y-b*(X-EX)
precio_control = np.mean(Y_control)

print("Sin variable de control:", round(y_media, 4))
print("Con variable de control:", round(precio_control, 4))

Sin variable de control: 5.9677
Con variable de control: 5.8293


### Comparación de varianzas

In [8]:
var_original = np.var(Y, ddof=1)
var_control = np.var(Y_control, ddof=1)

print("Varianza original:", round(var_original, 4))
print("Varianza con control:", round(var_control, 4))
print("Reducción:", round((1-var_control/var_original)*100, 2), "%")

Varianza original: 76.2941
Varianza con control: 0.044
Reducción: 99.94 %


Con la variable de control obtuve una estimación con menor varianza usando las mismas 100 rutas.